In [ ]:
from pathlib import Path
import os

# Must be set before importing torch.
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import json
from dataclasses import dataclass
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
from plyfile import PlyData
import viser
from nerfstudio.utils.eval_utils import eval_setup
from pytorch3d.transforms import quaternion_to_matrix

REPO_ROOT = Path("/robodata/smodak/repos/f3rm")
DATA_DIR = REPO_ROOT / "datasets/f3rm/fresh/objaverse/car2new"
CONFIG_PATH = REPO_ROOT / "mar06_outputs/car2new/f3rm/2026-03-06_164050/config.yml"
IMGPATH = DATA_DIR / "images/frame_00057.png"

SAM3_FEATURE_NAME = "sam3_"
SAM3D_FEATURE_NAME = "sam3d_"
MIN_INSTANCE_PIXELS = 500
MAX_VIS_POINTS = 120_000
VISER_PORT = 8890
ENABLE_PUBLIC_VISER = True
NERF_POINT_SIZE = 0.0035
SAM3D_MODEL_POINT_SIZE = 0.0012
NORM_ORD = 2
DO_SIMPLER_RATIO_OPTIM = False
RATIO_MAD_SIGMA_THR = 10.0
RESIDUAL_MAD_SIGMA_THR = 3.0
MIN_FIT_INLIERS = 300
FILTER_PLOT_BINS = 80

assert CONFIG_PATH.exists(), CONFIG_PATH
assert IMGPATH.exists(), IMGPATH
assert (DATA_DIR / "features" / SAM3_FEATURE_NAME).exists()
assert (DATA_DIR / "features" / SAM3D_FEATURE_NAME).exists()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("config:", CONFIG_PATH)
print("image:", IMGPATH)


In [ ]:
@dataclass
class CalibrationRow:
    instance_id: int
    mask_index: int
    n_pixels: int
    n_ratio_inliers: int
    n_final_inliers: int
    k_i: float
    mse_before: float
    mse_after: float
    median_before: float
    median_after: float


# Quick conversions from conventions in the gist quick-conversions section.
FACTOR_NERF_GL_TO_OPENCV = [1.0, -1.0, -1.0]
FACTOR_SAM3D_P3D_TO_OPENCV = [-1.0, -1.0, 1.0]
VISER_SERVER = None


def resolve_repo_path(path_like: str | Path, repo_root: Path = REPO_ROOT) -> Path:
    path = Path(path_like)
    return path if path.is_absolute() else (repo_root / path).resolve()


def apply_axis_multipliers(
    points: np.ndarray | torch.Tensor,
    factors_xyz: list[float],
) -> np.ndarray | torch.Tensor:
    if len(factors_xyz) != 3:
        raise ValueError(f"Expected 3 multipliers, got {factors_xyz}")
    if isinstance(points, torch.Tensor):
        scale = torch.tensor(factors_xyz, dtype=points.dtype, device=points.device)
        return points * scale
    scale_np = np.asarray(factors_xyz, dtype=points.dtype if isinstance(points, np.ndarray) else np.float32)
    return points * scale_np


def sam3d_p3d_to_opencv_camera(points: np.ndarray) -> np.ndarray:
    return apply_axis_multipliers(points, FACTOR_SAM3D_P3D_TO_OPENCV)


def build_feature_index_map(data_dir: Path, sam3d_feature_name: str) -> dict[Path, int]:
    meta_path = data_dir / "features" / sam3d_feature_name / "meta.pt"
    meta = torch.load(meta_path, map_location="cpu", weights_only=False)
    return {(REPO_ROOT / Path(rel)).resolve(): idx for idx, rel in enumerate(meta["image_fnames"])}


def find_nerf_image(pipeline: Any, image_path: Path) -> tuple[str, Any, int]:
    image_abs = image_path.resolve()
    for split in ("train", "eval"):
        dataset = getattr(pipeline.datamanager, f"{split}_dataset")
        index_by_path = {Path(p).resolve(): i for i, p in enumerate(dataset.image_filenames)}
        if image_abs in index_by_path:
            return split, dataset, index_by_path[image_abs]
    raise ValueError(f"IMGPATH not found in Nerfstudio train/eval datasets: {image_abs}")


def render_nerf_pointmap_camera(
    pipeline: Any,
    dataset: Any,
    image_idx: int,
    target_device: torch.device,
) -> dict[str, Any]:
    camera = dataset.cameras[image_idx : image_idx + 1].to(target_device)
    ray_bundle = camera.generate_rays(camera_indices=0, keep_shape=True).to(target_device)

    # Do not wrap in no_grad/inference_mode: predict_normals=True models may require grads.
    outputs = pipeline.model.get_outputs_for_camera_ray_bundle(ray_bundle, render_features=False)

    depth_ray = outputs["depth"].squeeze(-1)
    pointmap_world = ray_bundle.origins + ray_bundle.directions * depth_ray.unsqueeze(-1)

    # Nerfstudio/OpenGL camera coords: +X right, +Y up, -Z forward.
    c2w = camera.camera_to_worlds[0]
    R = c2w[:, :3]
    t = c2w[:, 3]
    pointmap_cam_gl = (pointmap_world - t) @ R

    # Convert to OpenCV camera coords: +X right, +Y down, +Z forward.
    pointmap_cam_cv = apply_axis_multipliers(pointmap_cam_gl, FACTOR_NERF_GL_TO_OPENCV)

    return {
        "camera": camera,
        "ray_bundle": ray_bundle,
        "outputs": outputs,
        "depthmap": pointmap_cam_cv[..., 2].detach().cpu().numpy().astype(np.float32),
        "pointmap_cam": pointmap_cam_cv.detach().cpu().numpy().astype(np.float32),
        "rgb": dataset.get_image_float32(image_idx).cpu().numpy().astype(np.float32),
    }


def load_sam3d_artifacts(
    data_dir: Path,
    feature_idx: int,
    sam3_feature_name: str,
    sam3d_feature_name: str,
) -> dict[str, Any]:
    sam3d_root = data_dir / "features" / sam3d_feature_name
    sam3_root = data_dir / "features" / sam3_feature_name

    entry_path = sam3d_root / f"image_{feature_idx:06d}.json"
    entry = json.loads(entry_path.read_text())

    manifest_path = resolve_repo_path(entry["manifest_path"])
    manifest = json.loads(manifest_path.read_text())

    pointmap_file = resolve_repo_path(manifest["pointmap_file"])
    # Saved by SAM3D compute_pointmap: camera-frame pointmap in SAM3D/PyTorch3D convention.
    pointmap_hwc_sam3d = np.load(pointmap_file)["pointmap_hwc"].astype(np.float32)

    sam3_npz_path = sam3_root / f"image_{feature_idx:06d}.npz"
    sam3_data = np.load(sam3_npz_path)
    masks = sam3_data["masks"]
    if masks.ndim == 4 and masks.shape[1] == 1:
        masks = masks[:, 0]
    elif masks.ndim != 3:
        raise ValueError(f"Unexpected SAM3 mask shape: {masks.shape}")

    return {
        "entry_path": entry_path,
        "entry": entry,
        "manifest_path": manifest_path,
        "manifest": manifest,
        "pointmap_hwc_sam3d": pointmap_hwc_sam3d,
        "masks": masks.astype(bool),
    }


def _mad_interval(values: np.ndarray, sigma_thresh: float, eps: float = 1e-8) -> tuple[float, float, float, float]:
    center = float(np.median(values))
    mad = float(np.median(np.abs(values - center)))
    sigma = max(1.4826 * mad, eps)
    lo = center - sigma_thresh * sigma
    hi = center + sigma_thresh * sigma
    return center, sigma, lo, hi


def _closed_form_scale_l2(sam3d_points: np.ndarray, nerf_points: np.ndarray, eps: float = 1e-12) -> float:
    denom = float(np.sum(sam3d_points * sam3d_points))
    if denom <= eps:
        raise ValueError("Degenerate SAM3D points: denominator is near zero")
    numer = float(np.sum(sam3d_points * nerf_points))
    return numer / denom


def fit_isotropic_scale_sam3d_to_nerf(
    sam3d_points: np.ndarray,
    nerf_points: np.ndarray,
    norm_ord: int | float = NORM_ORD,
    ratio_sigma_thr: float = RATIO_MAD_SIGMA_THR,
    residual_sigma_thr: float = RESIDUAL_MAD_SIGMA_THR,
    min_inliers: int = MIN_FIT_INLIERS,
    do_simpler_ratio_optim: bool = DO_SIMPLER_RATIO_OPTIM,
) -> tuple[float, float, float, float, float, dict[str, Any]]:
    if sam3d_points.shape != nerf_points.shape or sam3d_points.ndim != 2 or sam3d_points.shape[1] != 3:
        raise ValueError(f"Expected matched Nx3 arrays, got {sam3d_points.shape} and {nerf_points.shape}")

    sam_norm = np.linalg.norm(sam3d_points, ord=norm_ord, axis=1)
    nerf_norm = np.linalg.norm(nerf_points, ord=norm_ord, axis=1)

    norm_valid = np.isfinite(sam_norm) & np.isfinite(nerf_norm) & (sam_norm > 1e-8) & (nerf_norm > 1e-8)
    if int(norm_valid.sum()) < min_inliers:
        raise ValueError("Not enough valid points after finite/norm filtering")

    sam_valid = sam3d_points[norm_valid]
    nerf_valid = nerf_points[norm_valid]
    ratio_values = nerf_norm[norm_valid] / np.clip(sam_norm[norm_valid], 1e-8, None)

    if do_simpler_ratio_optim:
        ratio_values = ratio_values[np.isfinite(ratio_values)]
        if ratio_values.size < min_inliers:
            raise ValueError("Not enough valid points for simple ratio optimization")

        k_final = float(np.median(ratio_values))

        diff_before = sam_valid - nerf_valid
        diff_after = (k_final * sam_valid) - nerf_valid

        mse_before = float(np.mean(np.sum(diff_before**2, axis=1)))
        mse_after = float(np.mean(np.sum(diff_after**2, axis=1)))
        median_before = float(np.median(np.linalg.norm(diff_before, ord=norm_ord, axis=1)))
        median_after = float(np.median(np.linalg.norm(diff_after, ord=norm_ord, axis=1)))

        residual_values = np.linalg.norm(diff_after, ord=norm_ord, axis=1)
        all_true_ratio = np.ones((ratio_values.shape[0],), dtype=bool)
        all_true_res = np.ones((residual_values.shape[0],), dtype=bool)

        debug = {
            "mode": "simple_ratio",
            "ratio_values": ratio_values.astype(np.float32),
            "ratio_inlier": all_true_ratio,
            "ratio_center": float(np.median(ratio_values)),
            "ratio_sigma": 0.0,
            "ratio_lo": float(np.min(ratio_values)),
            "ratio_hi": float(np.max(ratio_values)),
            "residual_values": residual_values.astype(np.float32),
            "residual_inlier": all_true_res,
            "residual_center": float(np.median(residual_values)),
            "residual_sigma": 0.0,
            "residual_lo": float(np.min(residual_values)),
            "residual_hi": float(np.max(residual_values)),
            "k_init": float(k_final),
            "n_input": int(sam3d_points.shape[0]),
            "n_norm_valid": int(norm_valid.sum()),
            "n_ratio_inliers": int(ratio_values.shape[0]),
            "n_final_inliers": int(residual_values.shape[0]),
        }
        return k_final, mse_before, mse_after, median_before, median_after, debug

    ratio_center, ratio_sigma, ratio_lo, ratio_hi = _mad_interval(ratio_values, ratio_sigma_thr)
    ratio_inlier = (ratio_values >= ratio_lo) & (ratio_values <= ratio_hi)
    if int(ratio_inlier.sum()) < min_inliers:
        raise ValueError("Not enough points after ratio-based pre-filter")

    sam_prefit = sam_valid[ratio_inlier]
    nerf_prefit = nerf_valid[ratio_inlier]

    k_init = _closed_form_scale_l2(sam_prefit, nerf_prefit)
    residual_values = np.linalg.norm((k_init * sam_prefit) - nerf_prefit, ord=norm_ord, axis=1)
    residual_center, residual_sigma, residual_lo, residual_hi = _mad_interval(residual_values, residual_sigma_thr)
    residual_inlier = (residual_values >= max(0.0, residual_lo)) & (residual_values <= residual_hi)
    if int(residual_inlier.sum()) < min_inliers:
        raise ValueError("Not enough points after residual-based post-filter")

    sam_inliers = sam_prefit[residual_inlier]
    nerf_inliers = nerf_prefit[residual_inlier]

    k_final = _closed_form_scale_l2(sam_inliers, nerf_inliers)

    diff_before = sam_inliers - nerf_inliers
    diff_after = (k_final * sam_inliers) - nerf_inliers

    mse_before = float(np.mean(np.sum(diff_before**2, axis=1)))
    mse_after = float(np.mean(np.sum(diff_after**2, axis=1)))
    median_before = float(np.median(np.linalg.norm(diff_before, ord=norm_ord, axis=1)))
    median_after = float(np.median(np.linalg.norm(diff_after, ord=norm_ord, axis=1)))

    debug = {
        "mode": "robust_two_stage",
        "ratio_values": ratio_values.astype(np.float32),
        "ratio_inlier": ratio_inlier.astype(bool),
        "ratio_center": float(ratio_center),
        "ratio_sigma": float(ratio_sigma),
        "ratio_lo": float(ratio_lo),
        "ratio_hi": float(ratio_hi),
        "residual_values": residual_values.astype(np.float32),
        "residual_inlier": residual_inlier.astype(bool),
        "residual_center": float(residual_center),
        "residual_sigma": float(residual_sigma),
        "residual_lo": float(residual_lo),
        "residual_hi": float(residual_hi),
        "k_init": float(k_init),
        "n_input": int(sam3d_points.shape[0]),
        "n_norm_valid": int(norm_valid.sum()),
        "n_ratio_inliers": int(ratio_inlier.sum()),
        "n_final_inliers": int(residual_inlier.sum()),
    }

    return k_final, mse_before, mse_after, median_before, median_after, debug


def load_transformed_sam3d_instance_points(instance_record: dict[str, Any]) -> np.ndarray:
    gs_local_path = resolve_repo_path(instance_record["gs_local_path"])
    pose_path = resolve_repo_path(instance_record["pose_path"])

    ply = PlyData.read(str(gs_local_path))["vertex"]
    points_local = np.stack(
        [
            np.asarray(ply["x"]),
            np.asarray(ply["y"]),
            np.asarray(ply["z"]),
        ],
        axis=1,
    ).astype(np.float32)

    pose_meta = json.loads(pose_path.read_text())
    quat_wxyz = np.asarray(pose_meta["rotation_wxyz_l2c"], dtype=np.float32).reshape(4)
    trans_l2c = np.asarray(pose_meta["translation_l2c"], dtype=np.float32).reshape(3)
    scale_l2c = np.asarray(pose_meta["scale_l2c"], dtype=np.float32).reshape(3)

    R_l2c = quaternion_to_matrix(torch.from_numpy(quat_wxyz).float().unsqueeze(0))[0].cpu().numpy().astype(np.float32)
    # gs_local.ply is object-local; pose_l2c explicitly maps local -> camera frame.
    points_cam_sam3d = (points_local * scale_l2c[None, :]) @ R_l2c + trans_l2c[None, :]

    # SAM3D/PyTorch3D camera convention -> OpenCV camera convention
    points_cam_cv = sam3d_p3d_to_opencv_camera(points_cam_sam3d).astype(np.float32)
    points_cam_cv = points_cam_cv[np.isfinite(points_cam_cv).all(axis=1)]
    return points_cam_cv


def downsample_points(points: np.ndarray, max_points: int, seed: int = 0) -> np.ndarray:
    if points.shape[0] <= max_points:
        return points
    rng = np.random.default_rng(seed)
    keep = rng.choice(points.shape[0], size=max_points, replace=False)
    return points[keep]


def print_calibration_table(rows: list[CalibrationRow]) -> None:
    header = (
        f"{'inst':>6} {'mask':>6} {'pixels':>10} {'n_ratio':>10} {'n_final':>10} "
        f"{'k_i':>10} {'mse_before':>14} {'mse_after':>14} {'med_before':>12} {'med_after':>12}"
    )
    print(header)
    print("-" * len(header))
    for row in rows:
        print(
            f"{row.instance_id:6d} {row.mask_index:6d} {row.n_pixels:10d} "
            f"{row.n_ratio_inliers:10d} {row.n_final_inliers:10d} "
            f"{row.k_i:10.6f} {row.mse_before:14.6f} {row.mse_after:14.6f} "
            f"{row.median_before:12.6f} {row.median_after:12.6f}"
        )


def maybe_request_public_url(server: viser.ViserServer, enable: bool = ENABLE_PUBLIC_VISER) -> str | None:
    if not enable:
        return None
    try:
        return server.request_share_url(verbose=True)
    except Exception as exc:
        print(f"Public Viser URL request failed: {exc}")
        return None





In [ ]:
config, pipeline, checkpoint_path, step = eval_setup(
    config_path=CONFIG_PATH,
    test_mode="test",
)

split, dataset, nerf_image_idx = find_nerf_image(pipeline, IMGPATH)
nerf_render = render_nerf_pointmap_camera(pipeline, dataset, nerf_image_idx, device)

feature_index_map = build_feature_index_map(DATA_DIR, SAM3D_FEATURE_NAME)
img_abs = IMGPATH.resolve()
if img_abs not in feature_index_map:
    raise ValueError(f"IMGPATH not found in {SAM3D_FEATURE_NAME}/meta.pt: {img_abs}")
feature_idx = feature_index_map[img_abs]

sam3d_bundle = load_sam3d_artifacts(
    DATA_DIR,
    feature_idx=feature_idx,
    sam3_feature_name=SAM3_FEATURE_NAME,
    sam3d_feature_name=SAM3D_FEATURE_NAME,
)

sam3d_pointmap_cv = sam3d_p3d_to_opencv_camera(sam3d_bundle["pointmap_hwc_sam3d"])
if sam3d_pointmap_cv.shape != nerf_render["pointmap_cam"].shape:
    raise RuntimeError(
        f"Pointmap shape mismatch: SAM3D {sam3d_pointmap_cv.shape} vs NeRF {nerf_render['pointmap_cam'].shape}"
    )

state = {
    "pipeline": pipeline,
    "checkpoint_path": checkpoint_path,
    "step": step,
    "split": split,
    "nerf_image_idx": nerf_image_idx,
    "feature_idx": feature_idx,
    "nerf_rgb": nerf_render["rgb"],
    "nerf_depth": nerf_render["depthmap"],
    "nerf_pointmap_cam": nerf_render["pointmap_cam"],
    "sam3_masks": sam3d_bundle["masks"],
    "sam3d_entry": sam3d_bundle["entry"],
    "sam3d_manifest": sam3d_bundle["manifest"],
    "sam3d_pointmap_cv": sam3d_pointmap_cv,
}

print("checkpoint:", checkpoint_path)
print("step:", step)
print("dataset split/image idx:", split, nerf_image_idx)
print("SAM3D feature idx:", feature_idx)
print("num SAM3 masks:", state["sam3_masks"].shape[0])
print("num SAM3D instances:", len(state["sam3d_manifest"]["instances"]))

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(state["nerf_rgb"])
plt.title("Input image")
plt.axis("off")

plt.subplot(1, 2, 2)
im = plt.imshow(state["nerf_depth"], cmap="turbo")
plt.title("NeRF depth (camera z)")
plt.axis("off")
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()




In [ ]:
calibration_instances: list[dict[str, Any]] = []

for instance_record in state["sam3d_manifest"]["instances"]:
    instance_id = int(instance_record["instance_id"])
    mask_index = int(instance_record["mask_index"])

    if mask_index >= state["sam3_masks"].shape[0]:
        continue

    mask = state["sam3_masks"][mask_index]
    valid = mask & np.isfinite(state["sam3d_pointmap_cv"]).all(axis=-1) & np.isfinite(state["nerf_pointmap_cam"]).all(axis=-1)

    nerf_pts = state["nerf_pointmap_cam"][valid]
    sam_pm_pts = state["sam3d_pointmap_cv"][valid]
    if nerf_pts.shape[0] < MIN_INSTANCE_PIXELS:
        continue

    try:
        k_i, mse_before, mse_after, med_before, med_after, fit_debug = fit_isotropic_scale_sam3d_to_nerf(
            sam_pm_pts,
            nerf_pts,
            norm_ord=NORM_ORD,
            ratio_sigma_thr=RATIO_MAD_SIGMA_THR,
            residual_sigma_thr=RESIDUAL_MAD_SIGMA_THR,
            min_inliers=MIN_FIT_INLIERS,
            do_simpler_ratio_optim=DO_SIMPLER_RATIO_OPTIM,
        )
    except ValueError:
        continue

    if not np.isfinite(k_i) or k_i <= 0.0:
        continue

    row = CalibrationRow(
        instance_id=instance_id,
        mask_index=mask_index,
        n_pixels=int(nerf_pts.shape[0]),
        n_ratio_inliers=int(fit_debug["n_ratio_inliers"]),
        n_final_inliers=int(fit_debug["n_final_inliers"]),
        k_i=float(k_i),
        mse_before=float(mse_before),
        mse_after=float(mse_after),
        median_before=float(med_before),
        median_after=float(med_after),
    )

    calibration_instances.append(
        {
            "row": row,
            "instance_record": instance_record,
            "valid_mask": valid,
            "nerf_points": nerf_pts,
            "fit_debug": fit_debug,
        }
    )

if len(calibration_instances) == 0:
    raise RuntimeError(
        "No valid instance for calibration on this image. Choose another IMGPATH with a SAM3D instance."
    )

rows = [item["row"] for item in calibration_instances]
if not all(row.median_after <= row.median_before + 1e-10 for row in rows):
    raise RuntimeError("Pointmap-based fit check failed: median_after > median_before for at least one instance")

k_image = float(np.mean([row.k_i for row in rows]))

state["calibration_instances"] = calibration_instances
state["calibration_rows"] = rows
state["k_image"] = k_image

print_calibration_table(rows)
print()
print(f"Final SAM3D->NeRF image scale k_image (mean over instances): {k_image:.6f}")
print(f"Optimization mode: {'simple_ratio' if DO_SIMPLER_RATIO_OPTIM else 'robust_two_stage'}")



In [ ]:
INSTANCE_TO_VIS = 0  # index in state["calibration_instances"]

if INSTANCE_TO_VIS >= len(state["calibration_instances"]):
    raise IndexError(f"INSTANCE_TO_VIS={INSTANCE_TO_VIS} out of range")

vis_item = state["calibration_instances"][INSTANCE_TO_VIS]
inst_row = vis_item["row"]
inst_record = vis_item["instance_record"]

nerf_instance_points = vis_item["nerf_points"].astype(np.float32)
sam3d_pointmap_instance_points = state["sam3d_pointmap_cv"][vis_item["valid_mask"]].astype(np.float32)
sam3d_pointmap_instance_points_scaled = (state["k_image"] * sam3d_pointmap_instance_points).astype(np.float32)
sam3d_model_points_raw = load_transformed_sam3d_instance_points(inst_record)
sam3d_model_points_scaled = (state["k_image"] * sam3d_model_points_raw).astype(np.float32)

nerf_vis = downsample_points(nerf_instance_points, MAX_VIS_POINTS, seed=1)
sam_pm_scaled_vis = downsample_points(sam3d_pointmap_instance_points_scaled, MAX_VIS_POINTS, seed=11)
sam_raw_vis = downsample_points(sam3d_model_points_raw, MAX_VIS_POINTS, seed=2)
sam_scaled_vis = downsample_points(sam3d_model_points_scaled, MAX_VIS_POINTS, seed=3)

state["vis_payload"] = {
    "instance_to_vis": INSTANCE_TO_VIS,
    "inst_row": inst_row,
    "nerf_vis": nerf_vis,
    "sam_pm_scaled_vis": sam_pm_scaled_vis,
    "sam_raw_vis": sam_raw_vis,
    "sam_scaled_vis": sam_scaled_vis,
}

if VISER_SERVER is not None:
    VISER_SERVER.stop()

VISER_SERVER = viser.ViserServer(port=VISER_PORT)
VISER_SERVER.scene.add_point_cloud(
    "/nerf_instance",
    points=nerf_vis,
    colors=(0.1, 0.35, 0.95),
    point_size=NERF_POINT_SIZE,
)
VISER_SERVER.scene.add_point_cloud(
    "/sam3d_pointmap_instance_scaled",
    points=sam_pm_scaled_vis,
    colors=(0.0, 0.0, 0.0),
    point_size=NERF_POINT_SIZE,
)
VISER_SERVER.scene.add_point_cloud(
    "/sam3d_raw",
    points=sam_raw_vis,
    colors=(0.95, 0.2, 0.2),
    point_size=SAM3D_MODEL_POINT_SIZE,
)
VISER_SERVER.scene.add_point_cloud(
    "/sam3d_scaled",
    points=sam_scaled_vis,
    colors=(0.1, 0.9, 0.2),
    point_size=SAM3D_MODEL_POINT_SIZE,
)

print(f"Overlay for instance_id={inst_row.instance_id}, mask_index={inst_row.mask_index}")
print(f"k_i={inst_row.k_i:.6f}, k_image(mean)={state['k_image']:.6f}, norm_ord={NORM_ORD}")
print("Blue=NeRF instance pointmap, Black=SAM3D calibrated pointmap instance, Red=SAM3D raw model, Green=SAM3D calibrated model")
print(f"Viser running: http://localhost:{VISER_PORT}")
VISER_SERVER

public_url = maybe_request_public_url(VISER_SERVER, enable=ENABLE_PUBLIC_VISER)
if public_url:
    print(f"Public Viser URL: {public_url}")



In [ ]:
rows = state["calibration_rows"]

print_calibration_table(rows)
print()
print(f"Final SAM3D->NeRF scale (k_image): {state['k_image']:.6f}")

mean_med_before = float(np.mean([r.median_before for r in rows]))
mean_med_after = float(np.mean([r.median_after for r in rows]))
mean_mse_before = float(np.mean([r.mse_before for r in rows]))
mean_mse_after = float(np.mean([r.mse_after for r in rows]))

print(f"Mean median residual: {mean_med_before:.6f} -> {mean_med_after:.6f}")
print(f"Mean MSE residual:    {mean_mse_before:.6f} -> {mean_mse_after:.6f}")



In [ ]:
FILTER_PLOT_INSTANCE = 0  # index in state["calibration_instances"]

if FILTER_PLOT_INSTANCE >= len(state["calibration_instances"]):
    raise IndexError(f"FILTER_PLOT_INSTANCE={FILTER_PLOT_INSTANCE} out of range")

plot_item = state["calibration_instances"][FILTER_PLOT_INSTANCE]
plot_row = plot_item["row"]
plot_dbg = plot_item["fit_debug"]
mode_tag = plot_dbg.get("mode", "unknown")

ratio_vals = plot_dbg["ratio_values"]
ratio_inliers = plot_dbg["ratio_inlier"]
res_vals = plot_dbg["residual_values"]
res_inliers = plot_dbg["residual_inlier"]

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

ax[0].hist(ratio_vals[~ratio_inliers], bins=FILTER_PLOT_BINS, color=(0.95, 0.3, 0.3), alpha=0.65, label="filtered")
ax[0].hist(ratio_vals[ratio_inliers], bins=FILTER_PLOT_BINS, color=(0.2, 0.4, 0.95), alpha=0.65, label="kept")
ax[0].axvline(plot_dbg["ratio_lo"], color="black", linestyle="--", linewidth=1.2, label="threshold")
ax[0].axvline(plot_dbg["ratio_hi"], color="black", linestyle="--", linewidth=1.2)
ax[0].axvline(plot_dbg["ratio_center"], color="black", linestyle="-", linewidth=1.2, label="median")
ax[0].set_title(f"Pre-filter: norm-ratio inliers ({plot_dbg['n_ratio_inliers']}/{plot_dbg['n_norm_valid']})")
ax[0].set_xlabel("ratio ||P_nerf||_2 / ||P_sam3d||_2")
ax[0].set_ylabel("count")
ax[0].legend()

ax[1].hist(res_vals[~res_inliers], bins=FILTER_PLOT_BINS, color=(0.95, 0.3, 0.3), alpha=0.65, label="filtered")
ax[1].hist(res_vals[res_inliers], bins=FILTER_PLOT_BINS, color=(0.2, 0.7, 0.25), alpha=0.65, label="kept")
ax[1].axvline(max(0.0, plot_dbg["residual_lo"]), color="black", linestyle="--", linewidth=1.2, label="threshold")
ax[1].axvline(plot_dbg["residual_hi"], color="black", linestyle="--", linewidth=1.2)
ax[1].axvline(plot_dbg["residual_center"], color="black", linestyle="-", linewidth=1.2, label="median")
ax[1].set_title(f"Post-filter: residual inliers ({plot_dbg['n_final_inliers']}/{plot_dbg['n_ratio_inliers']})")
ax[1].set_xlabel("residual ||k0 * P_sam3d - P_nerf||_2")
ax[1].set_ylabel("count")
ax[1].legend()

plt.suptitle(
    f"Instance {plot_row.instance_id} (mask {plot_row.mask_index}) | mode={mode_tag} | k_init={plot_dbg['k_init']:.6f}, k_final={plot_row.k_i:.6f}",
    y=1.03,
)
plt.tight_layout()
plt.show()

